# Platt Lorrain Fine-Tuning with QLoRA

This notebook fine-tunes Mistral 7B to speak Platt Lorrain using QLoRA.

**Requirements:**
- Google Colab with T4 GPU (free tier works!)
- Upload `platt_chat_train.jsonl` to Colab

**Steps:**
1. Install dependencies
2. Load model with 4-bit quantization
3. Fine-tune with LoRA
4. Export adapter for Together AI

In [ ]:
# Install Unsloth (2x faster than standard training)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load Mistral 7B with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print("Model loaded!")

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters added!")

In [ ]:
# Upload your training data
from google.colab import files
uploaded = files.upload()  # Upload platt_chat_train.jsonl

In [ ]:
from datasets import load_dataset

# Load training data
dataset = load_dataset("json", data_files="platt_chat_train.jsonl", split="train")
print(f"Loaded {len(dataset)} training examples")
print("\nSample:")
print(dataset[0])

In [ ]:
# Format for Mistral Instruct
def format_chat(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print("\nFormatted sample:")
print(dataset[0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,  # 3 epochs for small dataset
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Trainer ready! Starting training...")

In [ ]:
# Train!
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# Test the model
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Du bist ein freundlicher Assistent, der Platt Lorrain spricht. Antworte immer auf Platt."},
    {"role": "user", "content": "Wie geht es dir heute?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=64,
    use_cache=True,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Save LoRA adapter for Together AI
model.save_pretrained("platt-lorrain-lora")
tokenizer.save_pretrained("platt-lorrain-lora")

print("Adapter saved to platt-lorrain-lora/")

In [ ]:
# Zip and download the adapter
!zip -r platt-lorrain-lora.zip platt-lorrain-lora/
files.download("platt-lorrain-lora.zip")

print("\n✅ Download complete!")
print("Next: Upload this to Together AI or Hugging Face Hub")

## Optional: Push to Hugging Face Hub

If you want to host on Hugging Face instead of Together AI:

In [ ]:
# Optional: Push to Hugging Face Hub
# from huggingface_hub import login
# login()  # Enter your HF token
# 
# model.push_to_hub("your-username/platt-lorrain-lora")
# tokenizer.push_to_hub("your-username/platt-lorrain-lora")